In [ ]:
# 04_evaluation.ipynb
# Goal of this notebook:
# - Load the trained models from notebook 03
# - Load the held-out test set (same for all models)
# - Compute test metrics: AUC (+ CI), accuracy, sensitivity, specificity
# - Save metrics into a table for the report
# - Optionally:
#    - Plot ROC curves
#    - Statistically compare models (paired tests on probabilities)
#
# You should:
# - Understand what each metric tells you in a medical context
# - Keep the test set "holy" (only used here, after training is done)

***

## 0. Setup and imports

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    auc,
    accuracy_score,
    confusion_matrix,
    recall_score,
)
from sklearn.calibration import calibration_curve
from scipy.stats import ttest_rel, norm
import pickle

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)

# 0.1 Set project root and move there
PROJECTROOT = Path.home() / "AIIP-radiomics-project"
os.chdir(PROJECTROOT)

RESULTS = PROJECTROOT / "results"
MODELDIR = RESULTS / "models"

print("Working directory:", os.getcwd())
print("Model directory:", MODELDIR)


def sensitivity_score(y_true, y_pred, pos_label=1):
    """Convenience wrapper for sensitivity (recall of the positive class)."""
    return recall_score(y_true, y_pred, pos_label=pos_label)


def specificity_score(y_true, y_pred, neg_label=0):
    """Convenience wrapper for specificity (recall of the negative class)."""
    return recall_score(y_true, y_pred, pos_label=neg_label)


print("✓ Setup complete")

**Student direction:**

- Ensure the same environment/kernel is used as for training, otherwise pickled models may fail to load.

***

## 1. Load trained models and test set

We now load:

- All `*.pkl` models in `results/models/` (from notebook 03).
- Test features and labels saved there as `.npy`.

In [ ]:
# 1.1 Load models
models = {}

# PSEUDOCODE:
# for model_file in MODELDIR.glob("*.pkl"):
#     model_name = model_file.stem
#     with open(model_file, "rb") as f:
#         models[model_name] = pickle.load(f)
#
# print(f"✓ Loaded {len(models)} models")
# print(" Models:", ", ".join(sorted(models.keys())))

# 1.2 Load test data
# (these were saved in notebook 03)
# PSEUDOCODE:
# X_raw_test = np.load(RESULTS / "Xtest_raw_top20.npy")
# y_test = np.load(RESULTS / "ytest.npy")
#
# print("\n✓ Test data loaded")
# print(" Test set size:", len(y_test))
# print(f" Positive class: {y_test.sum()} ({np.mean(y_test):.1%})")

**Student direction:**

- Confirm that the number of models matches what you trained (e.g. `lr_raw_binary_centers`, `rf_raw_binary_centers`, `gb_raw_binary_centers`).
- Confirm that `X_raw_test.shape[^9_0] == len(y_test)`.

***

## 2. AUC with confidence intervals (Hanley–McNeil approximation)

This helper computes AUC and an approximate confidence interval.

In [ ]:
def compute_auc_ci(y_true, y_pred_proba, alpha=0.05):
    """
    Compute AUC with a (1-alpha) confidence interval using the Hanley–McNeil approximation.

    Parameters:
        y_true: array of true binary labels (0/1)
        y_pred_proba: array of predicted probabilities for the positive class
        alpha: significance level (0.05 -> 95% CI)

    Returns:
        auc_score, ci_low, ci_high
    """
    auc_score = roc_auc_score(y_true, y_pred_proba)

    n_pos = np.sum(y_true)
    n_neg = len(y_true) - n_pos

    if n_pos == 0 or n_neg == 0:
        return auc_score, np.nan, np.nan

    q1 = auc_score / (2 - auc_score)
    q2 = 2 * auc_score**2 / (1 + auc_score)

    var = (
        auc_score * (1 - auc_score)
        + (n_pos - 1) * (q1 - auc_score**2)
        + (n_neg - 1) * (q2 - auc_score**2)
    ) / (n_pos * n_neg)

    var = max(var, 0.0)
    se = np.sqrt(var)

    z = norm.ppf(1 - alpha / 2)
    ci_low = max(0.0, auc_score - z * se)
    ci_high = min(1.0, auc_score + z * se)

    return auc_score, ci_low, ci_high

**Student direction:**

- You don’t need to edit this function; you will call it in the main evaluation loop.

***

## 3. Evaluate all models on the test set

For each model, we compute:

- AUC + CI
- Accuracy
- Sensitivity, specificity
- Store per-case probabilities for later ROC and statistical tests.

In [ ]:
# PSEUDOCODE:
# results = []
#
# for model_name, model in models.items():
#     X_test = X_raw_test
#
#     # 3.1 Predicted probabilities and hard labels
#     y_pred_proba = model.predict_proba(X_test)[:, 1]
#     y_pred = model.predict(X_test)
#
#     # Some models might output labels {1,2}; convert to {0,1}
#     y_pred_bin = np.where(y_pred == 2.0, 1, 0)
#
#     # 3.2 Metrics
#     auc_score, ci_low, ci_high = compute_auc_ci(y_test, y_pred_proba)
#     accuracy = accuracy_score(y_test, y_pred_bin)
#
#     cm = confusion_matrix(y_test, y_pred_bin)
#     tn, fp, fn, tp = cm.ravel()
#
#     sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
#     specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
#
#     results.append(
#         {
#             "model": model_name,
#             "auc": auc_score,
#             "auc_ci_low": ci_low,
#             "auc_ci_high": ci_high,
#             "accuracy": accuracy,
#             "sensitivity": sensitivity,
#             "specificity": specificity,
#             "y_pred_proba": y_pred_proba,  # keep for ROC/ttest
#         }
#     )
#
# # 3.3 Save a “clean” table without the probability arrays
# results_display = pd.DataFrame(
#     [{k: v for k, v in r.items() if k != "y_pred_proba"} for r in results]
# )
# results_display.to_csv(RESULTS / "evaluation_metrics.csv", index=False)
#
# print("\n" + "=" * 80)
# print("TEST SET PERFORMANCE METRICS")
# print("=" * 80)
# print(results_display.to_string(index=False))

**Student direction:**

- Inspect the printed table:
    - Which model has the highest AUC?
    - How do sensitivity and specificity compare?
- Remember that in medicine, sensitivity vs specificity trade‑offs matter, not just accuracy.

***

## 4. Confusion matrix and probability inspection for a reference model

It can be helpful to look closely at one model (e.g. logistic regression) to see how it behaves.

In [ ]:
# PSEUDOCODE:
# ref_model_name = "lr_raw_binary_centers"   # or another
# cm = confusion_matrix(y_test, models[ref_model_name].predict(X_raw_test))
# print("Confusion matrix for", ref_model_name)
# print(cm)
#
# probs_pos = models[ref_model_name].predict_proba(X_raw_test)[:, 1]
# print("Min/median/max predicted prob:", probs_pos.min(), np.median(probs_pos), probs_pos.max())
# print("Max prob among true positives:", probs_pos[y_test == 1].max())

**Student direction:**

- Use the confusion matrix to compute by hand: TP, FP, TN, FN.
- Reflect: are there many false negatives (missed center 2) or false positives?

***

## 5. ROC curves for all models on the test set

We now visualize ROC curves for each model to compare their discrimination.

In [ ]:
def plot_roc_curves(results, y_test):
    """
    Plot ROC curves for each model stored in `results`.
    """
    plt.figure(figsize=(7, 6))

    for r in results:
        model_name = r["model"]
        y_pred_proba = r["y_pred_proba"]
        fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{model_name} (AUC = {roc_auc:.3f})")

    # Chance line
    plt.plot([0, 1], [0, 1], "k--", label="Chance")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC curves on test set")
    plt.legend(loc="lower right", fontsize=9)
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

**Student direction:**

- Call `plot_roc_curves(results, y_test)` once you have the `results` list.
- Notice which curve is furthest towards the top‑left corner (better).

***

## 6. Optional: Paired statistical comparison of models

We can use paired tests on predicted probabilities to ask whether AUC differences might be due to chance. A more rigorous approach uses bootstrapping, but here’s a simple paired t‑test scaffold.

In [ ]:
def compare_models_paired_ttest(results, model_ids=None, alpha=0.05):
    """
    Paired t-tests on per-case predicted probabilities between models.

    Parameters:
        results: list of dicts (as above), each with keys 'model' and 'y_pred_proba'
        model_ids: list of model names to compare; if None, pick three standard ones
        alpha: global significance level (Bonferroni correction applied)
    """
    # Build a dict for easy lookup
    model_info = {r["model"]: r for r in results}

    if model_ids is None:
        # Example default: three models used in notebook 03
        model_ids = ["lr_raw_binary_centers", "rf_raw_binary_centers", "gb_raw_binary_centers"]

    # Sanity check: all present
    missing = [m for m in model_ids if m not in model_info]
    if missing:
        print("Missing models in results:", missing)
        return

    pairs = [
        (model_ids[^9_0], model_ids[^9_1]),
        (model_ids[^9_0], model_ids[^9_2]),
        (model_ids[^9_1], model_ids[^9_2]),
    ]

    alpha_pairwise = alpha / len(pairs)
    print("=" * 80)
    print("STATISTICAL COMPARISON (paired t-tests on predicted probabilities)")
    print(f" Global alpha = {alpha}, pairwise alpha (Bonferroni) = {alpha_pairwise:.4f}")
    print("=" * 80)

    rows = []

    for m1, m2 in pairs:
        r1 = model_info[m1]
        r2 = model_info[m2]

        auc1 = r1["auc"]
        auc2 = r2["auc"]

        ypred1 = r1["y_pred_proba"]
        ypred2 = r2["y_pred_proba"]

        tstat, pvalue = ttest_rel(ypred1, ypred2)

        print(f"{m1} vs {m2}")
        print(f" {m1} AUC: {auc1:.4f}")
        print(f" {m2} AUC: {auc2:.4f}")
        print(f" AUC difference (m2 - m1): {auc2 - auc1:.4f}")
        print(f" t-statistic: {tstat:.4f}")
        print(f" p-value: {pvalue:.4f} {'(significant)' if pvalue < alpha_pairwise else '(n.s.)'}")
        print()

        rows.append(
            {
                "model1": m1,
                "model2": m2,
                "auc1": auc1,
                "auc2": auc2,
                "auc_diff_m2_minus_m1": auc2 - auc1,
                "tstat": tstat,
                "pvalue": pvalue,
                "significant_bonferroni": "Yes" if pvalue < alpha_pairwise else "No",
            }
        )

    stats_df = pd.DataFrame(rows)
    stats_df.to_csv(RESULTS / "statistical_tests_models.csv", index=False)
    print("Saved statistical comparison table to:", RESULTS / "statistical_tests_models.csv")

**Student direction:**

- Use this only if you want to comment on whether AUC differences are likely to be due to chance.
- Remember that statistical significance is not the same as clinical importance.

***

## 7. Optional: Calibration plot for one model

Calibration shows whether predicted probabilities match observed frequencies.

In [ ]:
def plot_calibration(model_name, model, X_test, y_test, n_bins=10):
    """
    Reliability plot (calibration curve) for one model.
    """
    prob_pos = model.predict_proba(X_test)[:, 1]
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test, prob_pos, n_bins=n_bins, strategy="uniform"
    )

    plt.figure(figsize=(6, 6))
    plt.plot(mean_predicted_value, fraction_of_positives, "s-", label=model_name)
    plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")

    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction of positives")
    plt.title(f"Calibration curve: {model_name}")
    plt.legend(loc="upper left")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

**Student direction:**

- Choose one model (e.g. best AUC) and call `plot_calibration(model_name, model, X_raw_test, y_test)`.
- Interpret whether the model tends to over‑ or under‑estimate risk.

***

## 8. Checklist for this evaluation notebook

By the end of `04_evaluation.ipynb`, you should have:

- A table `results/evaluation_metrics.csv` with AUC (+ CI), accuracy, sensitivity, and specificity for all models.
- ROC curves showing which model discriminates best.
- Optionally, a `results/statistical_tests_models.csv` file with paired tests between models.
- One or two figures (e.g. ROC and calibration) and a short interpretation you can reuse in your final report.

[^9_1]: 04_evaluation.ipynb